# Movie Expert Agent

This notebook implements a Movie Expert Agent that answers questions about movies using the Nomad Movies API and OpenAI's `gpt-4o-mini` model with function calling.

In [ ]:
import os
import json
import httpx
from openai import OpenAI
from typing import List, Dict, Any, Optional
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

api_key = os.environ.get("OPENAI_API_KEY")
if not api_key:
    print("Error: OPENAI_API_KEY environment variable is not set.")
    print("Please check your .env file.")
else:
    client = OpenAI(api_key=api_key)
    print("OpenAI Client initialized successfully.")

## Movie API Wrapper

In [ ]:
BASE_URL = "https://nomad-movies.nomadcoders.workers.dev"

def get_popular_movies() -> List[Dict[str, Any]]:
    """Fetches popular movies from /movies."""
    response = httpx.get(f"{BASE_URL}/movies")
    response.raise_for_status()
    return response.json()

def get_movie_details(movie_id: int) -> Dict[str, Any]:
    """Fetches movie info from /movies/{id}."""
    response = httpx.get(f"{BASE_URL}/movies/{movie_id}")
    response.raise_for_status()
    return response.json()

def get_movie_credits(movie_id: int) -> Dict[str, Any]:
    """Fetches cast & crew from /movies/{id}/credits."""
    response = httpx.get(f"{BASE_URL}/movies/{movie_id}/credits")
    response.raise_for_status()
    return response.json()

def get_similar_movies(movie_id: int) -> List[Dict[str, Any]]:
    """Fetches similar movies from /movies/{id}/similar."""
    response = httpx.get(f"{BASE_URL}/movies/{movie_id}/similar")
    response.raise_for_status()
    return response.json()

## Agent Configuration

In [ ]:
MODEL = "gpt-4o-mini"

TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_popular_movies",
            "description": "Get a list of popular movies currently showing or trending.",
            "parameters": {
                "type": "object",
                "properties": {},
                "required": [],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_details",
            "description": "Get detailed information about a specific movie by its ID.",
            "parameters": {
                "type": "object",
                "properties": {
                    "movie_id": {
                        "type": "integer",
                        "description": "The ID of the movie to retrieve details for.",
                    },
                },
                "required": ["movie_id"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_credits",
            "description": "Get the cast and crew (credits) for a specific movie by its ID.",
            "parameters": {
                "type": "object",
                "properties": {
                    "movie_id": {
                        "type": "integer",
                        "description": "The ID of the movie to retrieve credits for.",
                    },
                },
                "required": ["movie_id"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_similar_movies",
            "description": "Get a list of movies similar to a specific movie by its ID.",
            "parameters": {
                "type": "object",
                "properties": {
                    "movie_id": {
                        "type": "integer",
                        "description": "The ID of the movie to find similar ones for.",
                    },
                },
                "required": ["movie_id"],
            },
        },
    },
]

In [ ]:
def run_conversation(user_prompt: str):
    # Step 1: send the conversation and available functions to the model
    messages = [
        {"role": "system", "content": "You are a Movie Expert Agent. Use the provided tools to answer user questions about movies."},
        {"role": "user", "content": user_prompt}
    ]
    
    print(f"\n[User]: {user_prompt}")
    
    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        tools=TOOLS,
        tool_choice="auto",
    )
    
    response_message = response.choices[0].message

    print(f"[Agent]: {response_message.content}")
    tool_calls = response_message.tool_calls
    
    # Step 2: check if the model wanted to call a function
    if tool_calls:
        # Step 3: call the function
        messages.append(response_message)
        
        for tool_call in tool_calls:
            function_name = tool_call.function.name
            function_args = json.loads(tool_call.function.arguments)
            
            print(f"[Agent]: Calling {function_name}({function_args})")
            
            try:
                if function_name == "get_popular_movies":
                    function_response = get_popular_movies()
                elif function_name == "get_movie_details":
                    function_response = get_movie_details(**function_args)
                elif function_name == "get_movie_credits":
                    function_response = get_movie_credits(**function_args)
                elif function_name == "get_similar_movies":
                    function_response = get_similar_movies(**function_args)
                else:
                    function_response = {"error": "Unknown function"}
            except Exception as e:
                function_response = {"error": str(e)}

            messages.append(
                {
                    "tool_call_id": tool_call.id,
                    "role": "tool",
                    "name": function_name,
                    "content": json.dumps(function_response),
                }
            )
        
        # Step 4: send the info for each function call and function response to the model
        second_response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
        )
        final_response = second_response.choices[0].message.content
        print(f"[Agent]: {final_response}")
        return final_response
    else:
        print(f"[Agent]: {response_message.content}")
        return response_message.content

## Interactive Chat

Run the cells below to interact with the agent.

In [ ]:
test_inputs = [
    "What are some popular movies right now?",
    "Tell me about movie ID 550",
    "Who stars in movie 550?"
]
for test_input in test_inputs:
    run_conversation(test_input)

In [ ]:
# Run this cell to chat interactive (Stop the cell to end)
while True:
    try:
        user_input = input("\nYou: ").strip()
        if user_input.lower() in ["exit", "quit"]:
            print("Ending conversation.")
            break
        if not user_input:
            continue
        run_conversation(user_input)
    except KeyboardInterrupt:
        print("\nConversation interrupted.")
        break
    except Exception as e:
        print(f"Error: {e}")